# BTC-ETH Pairs Trading — Full Analysis

**Pipeline**
1. Load & visualise close prices
2. ADF / KPSS stationarity tests
3. Johansen cointegration test
4. VECM fitting & ECT extraction
5. Z-Score signal generation
6. Backtest & performance metrics

All heavy lifting is delegated to the `src/` modules — this notebook is **exploration + presentation only**.

In [ ]:
import sys
from pathlib import Path

# Add project root to path
PROJECT_ROOT = Path("__file__").resolve().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams["figure.dpi"] = 120

In [ ]:
from src.config        import DATA_DIR, RESULTS_DIR
from src.data          import DataLoader
from src.preprocessing import log_transform, align_and_clean
from src.stats         import adf_test, kpss_test, stationarity_report, johansen_test
from src.models        import fit_vecm, get_ect, vecm_summary
from src.trading       import compute_spread, compute_zscore, generate_signals, run_backtest
from src.trading.backtest import metrics_table
from src.viz           import (
    plot_prices, plot_log_prices, plot_spread,
    plot_zscore_signals, plot_equity_curve, plot_stationarity_summary,
)

## 0. Parameters

In [ ]:
SYM_A    = "BTCUSDT"
SYM_B    = "ETHUSDT"
INTERVAL = "1h"

ZSCORE_ENTRY  = 2.0
ZSCORE_EXIT   = 0.0
ZSCORE_WINDOW = None   # None = full-sample stats

## 1. Load & Visualise Prices

In [ ]:
loader  = DataLoader(data_dir=DATA_DIR, interval=INTERVAL)
close_a = loader.load(SYM_A)
close_b = loader.load(SYM_B)

close  = align_and_clean({SYM_A: close_a, SYM_B: close_b})
log_px = log_transform(close)

print(f"Aligned samples: {len(close):,}")
print(f"Period: {close.index[0].date()} → {close.index[-1].date()}")
close.head()

In [ ]:
fig, ax = plot_prices(close, title=f"Close Prices — {SYM_A} / {SYM_B}")
plt.show()

In [ ]:
fig, ax = plot_log_prices(close, title=f"Log Prices — {SYM_A} / {SYM_B}")
plt.show()

## 2. Stationarity Tests

We expect log prices to be **I(1)**: non-stationary in levels, stationary in first differences.

In [ ]:
print("=== Log Prices (levels) ===")
report_level = stationarity_report(log_px)
display(report_level)

print("\n=== First Differences ===")
report_diff = stationarity_report(log_px.diff().dropna())
display(report_diff)

In [ ]:
fig, ax = plot_stationarity_summary(report_level,
    title=f"ADF p-values (levels) — {SYM_A} / {SYM_B}")
plt.show()

## 3. Johansen Cointegration Test

Tests whether a linear combination of the two I(1) series is I(0).

In [ ]:
j_result = johansen_test(log_px)
print(j_result)

johansen_df = pd.DataFrame({
    "Trace Statistic":   j_result.trace_stats,
    "95% Critical Value": j_result.trace_crit_95,
    "MaxEig Statistic":  j_result.max_eig_stats,
    "95% Critical Value (MaxEig)": j_result.max_eig_crit_95,
}, index=[f"r ≤ {i}" for i in range(len(j_result.trace_stats))])

display(johansen_df.round(4))

## 4. VECM — Fit & ECT Extraction

In [ ]:
vecm = fit_vecm(log_px)
print(vecm)
print()

summary = vecm_summary(vecm)
display(pd.Series(summary).rename("Value").to_frame())

In [ ]:
ect = get_ect(vecm, log_px)

fig, ax = plot_spread(ect, title=f"ECT Spread — {SYM_A} / {SYM_B}")
plt.show()

## 5. Z-Score & Trading Signals

| Signal | Condition | Trade |
|--------|-----------|-------|
| +1 (Long spread)  | Z < −2.0 | Long BTC / Short ETH |
| −1 (Short spread) | Z > +2.0 | Short BTC / Long ETH |
| 0  (Flat)         | \|Z\| < 0 | Close position |

In [ ]:
zscore  = compute_zscore(ect, window=ZSCORE_WINDOW)
signals = generate_signals(zscore, entry=ZSCORE_ENTRY, exit_=ZSCORE_EXIT)

print("Position distribution:")
print(signals.value_counts().sort_index())

In [ ]:
fig, ax = plot_zscore_signals(
    zscore, signals=signals,
    entry=ZSCORE_ENTRY, exit_=ZSCORE_EXIT,
    title=f"Z-Score Signals — {SYM_A} / {SYM_B}",
)
plt.show()

## 6. Backtest & Performance

In [ ]:
bt = run_backtest(
    signals       = signals,
    log_price_a   = log_px[SYM_A],
    log_price_b   = log_px[SYM_B],
    beta          = vecm.ect_coef,
)

display(metrics_table(bt))

In [ ]:
fig, axes = plot_equity_curve(
    bt.equity_curve,
    metrics=bt.metrics,
    title=f"Equity Curve — {SYM_A} / {SYM_B}",
)
plt.show()

## 7. Save All Results

In [ ]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
tag = f"{SYM_A}_{SYM_B}_{INTERVAL}"

plot_log_prices(close,   save_path=RESULTS_DIR / f"{tag}_log_prices.png")
plot_spread(ect,         save_path=RESULTS_DIR / f"{tag}_spread.png")
plot_zscore_signals(zscore, signals=signals,
                         save_path=RESULTS_DIR / f"{tag}_zscore.png")
plot_equity_curve(bt.equity_curve, metrics=bt.metrics,
                         save_path=RESULTS_DIR / f"{tag}_equity.png")

bt.equity_curve.to_csv(RESULTS_DIR / f"{tag}_equity.csv")
pd.DataFrame(bt.metrics, index=[0]).to_csv(RESULTS_DIR / f"{tag}_metrics.csv", index=False)

print("Done. All results saved to", RESULTS_DIR)